In [1]:
#Install required libraries in Colab
!pip install -q torchinfo einops

#Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchinfo import summary
import time
from einops import rearrange
import numpy as np
import math
import random

#Set seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

#Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Transform and Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
])

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


100%|██████████| 169M/169M [00:02<00:00, 81.0MB/s]


In [2]:
###efine Vision Transformer (ViT)
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=4, emb_dim=256, img_size=32):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 1 + self.n_patches, emb_dim))

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)  # (B, emb_dim, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)  # (B, n_patches, emb_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, emb_dim)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, 1 + n_patches, emb_dim)
        return x + self.pos_embed

class Attention(nn.Module):
    def __init__(self, dim, heads=2):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)
        scores = (q @ k.transpose(-2, -1)) * self.scale
        attn = scores.softmax(dim=-1)
        out = (attn @ v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.proj(out)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=2, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 num_classes=100, emb_dim=256, depth=4, heads=2, mlp_ratio=2):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, emb_dim, img_size)
        self.encoder = nn.Sequential(
            *[TransformerEncoderBlock(emb_dim, heads, mlp_ratio) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.encoder(x)
        x = self.norm(x)
        return self.head(x[:, 0])


In [3]:
#####Train and Evaluate
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


In [4]:
config = {
    'patch_size': 4,
    'emb_dim': 256,
    'depth': 4,
    'heads': 2,
    'mlp_ratio': 2
}

model = ViT(patch_size=config['patch_size'],
            emb_dim=config['emb_dim'],
            depth=config['depth'],
            heads=config['heads'],
            mlp_ratio=config['mlp_ratio']).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()
for epoch in range(10):  # you can increase to 20–50
    loss = train(model, train_loader, optimizer, criterion)
    acc = evaluate(model, test_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Test Acc = {acc:.4f}")
train_time = time.time() - start_time


Epoch 1: Loss = 3.8313, Test Acc = 0.1438
Epoch 2: Loss = 3.4835, Test Acc = 0.1809
Epoch 3: Loss = 3.3902, Test Acc = 0.1702
Epoch 4: Loss = 3.4764, Test Acc = 0.1527
Epoch 5: Loss = 3.4982, Test Acc = 0.1647
Epoch 6: Loss = 3.4306, Test Acc = 0.1493
Epoch 7: Loss = 3.4494, Test Acc = 0.1644
Epoch 8: Loss = 3.4042, Test Acc = 0.1740
Epoch 9: Loss = 3.3758, Test Acc = 0.1707
Epoch 10: Loss = 3.4691, Test Acc = 0.1662


In [5]:
summary(model, input_size=(1, 3, 32, 32))  # For parameter and layer info


Layer (type:depth-idx)                   Output Shape              Param #
ViT                                      [1, 100]                  --
├─PatchEmbedding: 1-1                    [1, 65, 256]              16,896
│    └─Conv2d: 2-1                       [1, 256, 8, 8]            12,544
├─Sequential: 1-2                        [1, 65, 256]              --
│    └─TransformerEncoderBlock: 2-2      [1, 65, 256]              --
│    │    └─LayerNorm: 3-1               [1, 65, 256]              512
│    │    └─Attention: 3-2               [1, 65, 256]              263,168
│    │    └─LayerNorm: 3-3               [1, 65, 256]              512
│    │    └─Sequential: 3-4              [1, 65, 256]              262,912
│    └─TransformerEncoderBlock: 2-3      [1, 65, 256]              --
│    │    └─LayerNorm: 3-5               [1, 65, 256]              512
│    │    └─Attention: 3-6               [1, 65, 256]              263,168
│    │    └─LayerNorm: 3-7               [1, 65, 256]      

In [6]:
#####Helper Functions to Log Results
results = []

def run_vit_experiment(config, name):
    print(f"\n🚀 Running {name}...")
    model = ViT(
        patch_size=config['patch_size'],
        emb_dim=config['emb_dim'],
        depth=config['depth'],
        heads=config['heads'],
        mlp_ratio=config['mlp_ratio']
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    start = time.time()
    for epoch in range(10):
        train_loss = train(model, train_loader, optimizer, criterion)
    end = time.time()

    test_acc = evaluate(model, test_loader)
    time_per_epoch = (end - start) / 10
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Collect result
    results.append({
        'Name': name,
        'Accuracy': round(test_acc * 100, 2),
        'Params (M)': round(param_count / 1e6, 2),
        'Epoch Time (s)': round(time_per_epoch, 2),
    })

    print(f" {name}: Acc={test_acc*100:.2f}%, Params={param_count/1e6:.2f}M, Time/Epoch={time_per_epoch:.2f}s")


In [7]:
###### Run All 4 ViT Configurations
configs = {
    "ViT-A": {'patch_size': 4, 'emb_dim': 256, 'depth': 4, 'heads': 2, 'mlp_ratio': 2},
    "ViT-B": {'patch_size': 4, 'emb_dim': 512, 'depth': 8, 'heads': 4, 'mlp_ratio': 4},
    "ViT-C": {'patch_size': 8, 'emb_dim': 256, 'depth': 4, 'heads': 4, 'mlp_ratio': 2},
    "ViT-D": {'patch_size': 8, 'emb_dim': 512, 'depth': 8, 'heads': 2, 'mlp_ratio': 4},
}

for name, cfg in configs.items():
    run_vit_experiment(cfg, name)



🚀 Running ViT-A...
✅ ViT-A: Acc=19.15%, Params=2.16M, Time/Epoch=8.83s

🚀 Running ViT-B...
✅ ViT-B: Acc=14.67%, Params=25.33M, Time/Epoch=38.46s

🚀 Running ViT-C...
✅ ViT-C: Acc=31.11%, Params=2.19M, Time/Epoch=9.28s

🚀 Running ViT-D...
✅ ViT-D: Acc=5.91%, Params=25.38M, Time/Epoch=16.73s


In [10]:
###########Baseline — ResNet-18
def run_resnet18():
    print("\n Running ResNet-18 baseline...")
    model = models.resnet18(weights=None, num_classes=100).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    start = time.time()
    for epoch in range(10):
        train_loss = train(model, train_loader, optimizer, criterion)
    end = time.time()

    test_acc = evaluate(model, test_loader)
    time_per_epoch = (end - start) / 10
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    results.append({
        'Name': 'ResNet-18',
        'Accuracy': round(test_acc * 100, 2),
        'Params (M)': round(param_count / 1e6, 2),
        'Epoch Time (s)': round(time_per_epoch, 2),
    })

    print(f" ResNet-18: Acc={test_acc*100:.2f}%, Params={param_count/1e6:.2f}M, Time/Epoch={time_per_epoch:.2f}s")

run_resnet18()



 Running ResNet-18 baseline...
 ResNet-18: Acc=43.80%, Params=11.23M, Time/Epoch=9.46s


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

df = pd.DataFrame(results)
display(df)


,Name,Accuracy,Params (M),Epoch Time (s)
0,ViT-A,19.15,2.16,8.83
1,ViT-B,14.67,25.33,38.46
2,ViT-C,31.11,2.19,9.28
3,ViT-D,5.91,25.38,16.73
4,ResNet-18,44.73,11.23,8.83
